<a href="https://colab.research.google.com/github/ngynth/Intra_Prediction_Mode_Analysis/blob/main/intra_codec_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies
!pip install -q streamlit opencv-python-headless numpy pandas matplotlib scikit-image seaborn

In [ ]:
import os

def convert_y4m_to_raw_yuv(input_path, output_path):
    # Ensure the directory exists (optional, but good practice)
    with open(input_path, 'rb') as f_in:
        data = f_in.read()
        header_end = data.find(b'\x0a')

        with open(output_path, 'wb') as f_out:
            f_out.write(data[header_end + 1:])

    print(f"Conversion complete. Raw file saved as: {output_path}")

# Define paths
input_file = 'Datasets/akiyo_qcif.y4m'
output_file = 'akiyo_qcif.yuv' # This will be saved in the root folder (current directory)

# Check if the file exists before converting
if os.path.exists(input_file):
    convert_y4m_to_raw_yuv(input_file, output_file)

    # Verify the file was created
    if os.path.exists(output_file):
        print(f"Success! File size: {os.path.getsize(output_file)} bytes")
else:
    print(f"Error: The file {input_file} was not found. Check your file path.")

In [ ]:
%%writefile intra_analysis.py
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from skimage.metrics import structural_similarity as ssim

class QualityMetricsCalculator:
    @staticmethod
    def compute_metrics(original, reconstructed, qp, standard):
        # Convert images to float for precise mathematical calculation
        orig = original.astype(np.float64)
        rec = reconstructed.astype(np.float64)

        # MSE measures pixel-wise distortion
        mse = np.mean((orig - rec) ** 2)
        # PSNR converts MSE into decibels; higher is better
        psnr = 100.0 if mse == 0 else 10 * np.log10((255.0 ** 2) / mse)
        # SSIM measures human-perceived structural degradation
        ssim_val = ssim(original, reconstructed, data_range=255)

        # VVC estimation: Bits are allocated based on QP (Quantization Parameter)
        base_bpp = 8.0
        # Higher QP = fewer bits (more compression, lower quality)
        compressed_bpp = base_bpp * (1.0 - (qp / 52.0))
        # VVC standard efficiency factor
        if standard == "VVC": compressed_bpp *= 0.85
        cr = base_bpp / max(compressed_bpp, 0.1)

        return psnr, ssim_val, cr, compressed_bpp

class IntraPredictionEngine:
    @staticmethod
    def stream_frames(yuv_path, num_frames=20):
        # YUV QCIF standard frame size
        width, height = 176, 144
        frame_size = int(width * height * 1.5)
        with open(yuv_path, 'rb') as f:
            for _ in range(num_frames):
                raw = f.read(frame_size)
                if len(raw) < frame_size: break
                yield raw

    @staticmethod
    def process_frame(raw_data, qp=32):
        width, height = 176, 144
        start_time = time.perf_counter()

        # Load raw binary into 8-bit grayscale array
        gray_frame = np.frombuffer(raw_data, dtype=np.uint8, count=width*height).reshape((height, width))
        reconstructed = np.zeros_like(gray_frame)

        # This map tracks the "size" of the block used at each location for visualization
        partition_map = np.zeros((height // 8, width // 8), dtype=np.int32)

        # Recursive function to mimic VVC's Coding Tree Structure (CTS)
        def split_block(y, x, h, w):
            block = gray_frame[y:y+h, x:x+w]
            # Decision: If the area is complex (high variance), we split to better preserve detail
            if np.var(block) > 500 and h > 16:
                h2, w2 = h // 2, w // 2
                # Recurse into 4 sub-quadrants (Quadtree partitioning)
                split_block(y, x, h2, w2)
                split_block(y, x + w2, h2, w2)
                split_block(y + h2, x, h2, w2)
                split_block(y + h2, x + w2, h2, w2)
            else:
                # Leaf node: This is where actual compression happens
                # We add noise proportional to QP to simulate lossy quantization
                noise = np.random.normal(0, (qp / 52.0) * 30.0, (h, w))
                reconstructed[y:y+h, x:x+w] = np.clip(block + noise, 0, 255)
                # Store block dimensions for the heatmap
                partition_map[y//8 : (y+h)//8, x//8 : (x+w)//8] = max(h, w)

        # Start the recursive partitioning process
        split_block(0, 0, height, width)

        latency = (time.perf_counter() - start_time) * 1000
        return gray_frame, reconstructed, partition_map, latency

if __name__ == "__main__":
    yuv_file = "akiyo_qcif.yuv"
    qp_values = [12, 22, 32, 42, 52]
    rd_data = []

    print("Starting Comprehensive Intra Analysis with VVC-style CTS partitioning...")

    for qp in qp_values:
        p_list, s_list, c_list, l_list = [], [], [], []
        frame_gen = IntraPredictionEngine.stream_frames(yuv_file, num_frames=5)

        for raw_frame in frame_gen:
            # Process frames through our recursive engine
            orig, rec, part_map, lat = IntraPredictionEngine.process_frame(raw_frame, qp)
            # Calculate metrics after compression simulation
            psnr, ssim_val, cr, bpp = QualityMetricsCalculator.compute_metrics(orig, rec, qp, "VVC")
            p_list.append(psnr); s_list.append(ssim_val); c_list.append(cr); l_list.append(lat)

        rd_data.append({"qp": qp, "psnr": np.mean(p_list), "bpp": bpp, "ssim": np.mean(s_list), "cr": np.mean(c_list), "lat": np.mean(l_list)})
        print(f"QP {qp:02d} | PSNR: {np.mean(p_list):.2f} dB | SSIM: {np.mean(s_list):.4f} | CR: {np.mean(c_list):.2f}:1 | Latency: {np.mean(l_list):.2f} ms")

    # Visualize the trade-off between Bitrate and Quality
    plt.figure(figsize=(8, 4))
    plt.plot([d['bpp'] for d in rd_data], [d['psnr'] for d in rd_data], 'ro-')
    plt.title("Rate-Distortion Curve (CTS Enabled)")
    plt.xlabel("Bitrate (bpp)"); plt.ylabel("PSNR (dB)"); plt.grid(True)
    plt.savefig("rd_curve.png")

    # Visualize how the encoder adaptively partitioned the frame
    plt.figure(figsize=(6, 5))
    sns.heatmap(part_map, annot=False, cmap="viridis")
    plt.title("Coding Tree Structure (CTS) Partitioning"); plt.xlabel("Block X"); plt.ylabel("Block Y")
    plt.savefig("cts_heatmap.png")
    print("\nReports saved: 'rd_curve.png' and 'cts_heatmap.png'.")

In [ ]:
!python intra_analysis.py